In [1]:
!pip install pyspark

from pyspark.sql import SparkSession
spark = SparkSession.builder.master("local[*]").getOrCreate()
sc = spark.sparkContext

In [2]:
!pip install wget
import wget

base = "https://raw.githubusercontent.com/nongaussian/class-2024-datamining/main/data/World Hapiness Report/"
years = [2015, 2016, 2017, 2018, 2019]

for y in years:
    wget.download(base + f"{y}.csv", f"{y}.csv")

In [3]:
#연도별 컬럼 확인
for y in years:
    print(y, sc.textFile(f"{y}.csv").first())
    print(sc.textFile(f"{y}.csv").take(3))
    print()

2015 Country,Happiness Rank,Happiness Score,Economy (GDP per Capita),Family,Health (Life Expectancy),Freedom,Trust (Government Corruption)
['Country,Happiness Rank,Happiness Score,Economy (GDP per Capita),Family,Health (Life Expectancy),Freedom,Trust (Government Corruption)', 'Switzerland,1,7.587,1.39651,1.34951,0.94143,0.66557,0.41978', 'Iceland,2,7.561,1.30232,1.40223,0.94784,0.62877,0.14145']

2016 Country,Happiness Rank,Happiness Score,Economy (GDP per Capita),Family,Health (Life Expectancy),Freedom,Trust (Government Corruption)
['Country,Happiness Rank,Happiness Score,Economy (GDP per Capita),Family,Health (Life Expectancy),Freedom,Trust (Government Corruption)', 'Denmark,1,7.526,1.44178,1.16374,0.79504,0.57941,0.44453', 'Switzerland,2,7.509,1.52733,1.14524,0.86303,0.58557,0.41203']

2017 Country,Happiness Rank,Happiness Score,Economy (GDP per Capita),Family,Health (Life Expectancy),Freedom,Generosity,Trust (Government Corruption)
['Country,Happiness Rank,Happiness Score,Economy (

In [4]:
#1. split(",")으로 자르기 전에, 칸 수가 헤더와 다른 줄이 있는지 확인
for y in years:
    rdd = sc.textFile(f"{y}.csv")
    n = len(rdd.first().split(","))
    print(y, rdd.filter(lambda x: len(x.split(",")) != n).collect())

#2017년 "Hong Kong S.A.R., China"는 이름 안에 쉼표가 있어서 split(",")을 하면 칸이 밀림.
#(나라, (연도,점수))로 변환할때 happiness score가 들어갈곳에 happiness Rank가 들어가는 불상사 발생.
#-> 오른쪽부터 (컬럼 수 - 1)번만 자르는 rsplit을 사용. 이름 양 끝의 따옴표는 제거.

2015 []
2016 []
2017 ['"Hong Kong S.A.R., China",71,5.472000122,1.551674843,1.262790918,0.943062425,0.490968645,0.374465793,0.293933749']
2018 []
2019 []


In [5]:
#각 줄을 (나라, (연도,점수)) 꼴로 변환
def load(y):
    rdd = sc.textFile(f"{y}.csv")
    header = rdd.first()
    n = len(header.split(","))

    return rdd.filter(lambda x: x != header) \
              .map(lambda x: x.rsplit(",", n-1)) \
              .map(lambda x: (x[0].strip('"'), (y, round(float(x[2]), 3)))) #나라[0]과 행복지수[2]만 꺼내 (나라, (연도,점수))꼴로 변환
                                                                            #행복지수는 소수점 반올림으로 통일.

print(load(2015).take(3))
print(load(2017).take(3))

[('Switzerland', (2015, 7.587)), ('Iceland', (2015, 7.561)), ('Denmark', (2015, 7.527))]
[('Norway', (2017, 7.537)), ('Denmark', (2017, 7.522)), ('Iceland', (2017, 7.504))]


In [6]:
#5개 년도를 union으로 합침
scores = load(2015)
for y in years[1:]:
    scores = scores.union(load(y))

print(scores.count())
print(scores.take(3))

782
[('Switzerland', (2015, 7.587)), ('Iceland', (2015, 7.561)), ('Denmark', (2015, 7.527))]


In [7]:
#2. 5년치가 다 없는 나라와, 그 나라가 나온 연도 확인
print(scores.groupByKey()
            .mapValues(lambda v: sorted([y for y, _ in v]))
            .filter(lambda x: len(x[1]) < 5)
            .sortByKey()
            .collect())
#비슷한 이름끼리 연도가 겹치지 않고 합치면 5년이 됨-> 같은나라인데 이름이 달라서 다르게 분류된것. 이름 통합하기

[('Angola', [2015, 2016, 2017, 2018]), ('Belize', [2016, 2017, 2018]), ('Central African Republic', [2015, 2017, 2018, 2019]), ('Comoros', [2015, 2016, 2019]), ('Djibouti', [2015]), ('Gambia', [2019]), ('Hong Kong', [2015, 2016, 2018, 2019]), ('Hong Kong S.A.R., China', [2017]), ('Laos', [2015, 2016, 2018, 2019]), ('Lesotho', [2015, 2017, 2018, 2019]), ('Macedonia', [2015, 2016, 2017, 2018]), ('Mozambique', [2015, 2017, 2018, 2019]), ('Namibia', [2016, 2017, 2018, 2019]), ('North Cyprus', [2015, 2016, 2017]), ('North Macedonia', [2019]), ('Northern Cyprus', [2018, 2019]), ('Oman', [2015]), ('Puerto Rico', [2016]), ('Somalia', [2016, 2017, 2018, 2019]), ('Somaliland Region', [2016]), ('Somaliland region', [2015]), ('South Sudan', [2016, 2017, 2018, 2019]), ('Sudan', [2015, 2016, 2017, 2018]), ('Suriname', [2015, 2016]), ('Swaziland', [2015, 2019]), ('Taiwan', [2015, 2016, 2018, 2019]), ('Taiwan Province of China', [2017]), ('Trinidad & Tobago', [2018, 2019]), ('Trinidad and Tobago', [20

In [8]:
name_map = {
    "Hong Kong S.A.R., China": "Hong Kong",
    "Taiwan Province of China": "Taiwan",
    "Macedonia": "North Macedonia",
    "Northern Cyprus": "North Cyprus",
    "Trinidad & Tobago": "Trinidad and Tobago",
    "Somaliland Region": "Somaliland region",
}
bc_name = sc.broadcast(name_map)

scores = scores.map(lambda x: (bc_name.value.get(x[0], x[0]), x[1]))

# 다시 확인: 이름 짝들이 사라져야 함
print(scores.groupByKey()
            .mapValues(lambda v: sorted([y for y, _ in v]))
            .filter(lambda x: len(x[1]) < 5)
            .sortByKey()
            .collect())

[('Angola', [2015, 2016, 2017, 2018]), ('Belize', [2016, 2017, 2018]), ('Central African Republic', [2015, 2017, 2018, 2019]), ('Comoros', [2015, 2016, 2019]), ('Djibouti', [2015]), ('Gambia', [2019]), ('Laos', [2015, 2016, 2018, 2019]), ('Lesotho', [2015, 2017, 2018, 2019]), ('Mozambique', [2015, 2017, 2018, 2019]), ('Namibia', [2016, 2017, 2018, 2019]), ('Oman', [2015]), ('Puerto Rico', [2016]), ('Somalia', [2016, 2017, 2018, 2019]), ('Somaliland region', [2015, 2016]), ('South Sudan', [2016, 2017, 2018, 2019]), ('Sudan', [2015, 2016, 2017, 2018]), ('Suriname', [2015, 2016]), ('Swaziland', [2015, 2019])]


In [9]:
#점수 한 번도 안 떨어진 나라만 찾기. 이름순 정렬
result = scores.groupByKey() \
    .mapValues(lambda x: [s for _, s in sorted(x)]) \
    .filter(lambda x: len(x[1]) == 5) \
    .filter(lambda x: all(x[1][i] <= x[1][i+1] for i in range(4))) \
    .sortByKey()

print(result.count())
print(result.take(3))

32
[('Benin', [3.34, 3.484, 3.657, 4.141, 4.883]), ('Burkina Faso', [3.587, 3.739, 4.032, 4.424, 4.587]), ('Burundi', [2.905, 2.905, 2.905, 2.905, 3.775])]


In [10]:
for country, s in result.collect():
    print(country, s)

Benin [3.34, 3.484, 3.657, 4.141, 4.883]
Burkina Faso [3.587, 3.739, 4.032, 4.424, 4.587]
Burundi [2.905, 2.905, 2.905, 2.905, 3.775]
Cambodia [3.819, 3.907, 4.168, 4.433, 4.7]
Cameroon [4.252, 4.513, 4.695, 4.975, 5.044]
Chad [3.667, 3.763, 3.936, 4.301, 4.35]
Congo (Brazzaville) [3.989, 4.236, 4.291, 4.559, 4.812]
Czech Republic [6.505, 6.596, 6.609, 6.711, 6.852]
Dominican Republic [4.885, 5.155, 5.23, 5.302, 5.425]
Estonia [5.429, 5.517, 5.611, 5.739, 5.893]
Finland [7.406, 7.413, 7.469, 7.632, 7.769]
Gabon [3.896, 4.121, 4.465, 4.758, 4.799]
Honduras [4.788, 4.871, 5.181, 5.504, 5.86]
Hungary [4.8, 5.145, 5.324, 5.62, 5.758]
Ivory Coast [3.655, 3.916, 4.18, 4.671, 4.944]
Latvia [5.098, 5.56, 5.85, 5.933, 5.94]
Malta [6.302, 6.488, 6.527, 6.627, 6.726]
Mongolia [4.874, 4.907, 4.955, 5.125, 5.285]
Niger [3.845, 3.856, 4.028, 4.166, 4.628]
North Macedonia [5.007, 5.121, 5.175, 5.185, 5.274]
Philippines [5.073, 5.279, 5.43, 5.524, 5.631]
Poland [5.791, 5.835, 5.973, 6.123, 6.182]
Port